# Diffusion Reimagined: The Mathematics of Flow Matching and Rectified Flows

> **Topic**: Generative AI, Continuous Normalizing Flows, Optimal Transport Displacement Interpolation  
> **Key Breakthrough**: Replacing curved stochastic SDE trajectories with straight-line vector fields for 1-step sampling.

---

## 1. The Geometry of Rectified Flows
Traditional Diffusion (DDPM/Score-based SDEs) follows Brownian motion diffusion curves, requiring 50–1000 iterative solver steps.

Flow Matching defines a continuous probability velocity field $v_t(x)$ that transports standard Gaussian noise $X_0 \sim \mathcal{N}(0, I)$ directly to the data distribution $X_1 \sim p_{\text{data}}$ along straight displacement lines:
$$X_t = (1 - t) X_0 + t X_1, \quad t \in [0, 1]$$

The instantaneous velocity is simply:
$$\frac{d X_t}{d t} = X_1 - X_0$$

The neural network $v_\theta(X_t, t)$ is trained via simple Mean Squared Error regression:
$$\mathcal{L}_{\text{FM}}(\theta) = \mathbb{E}_{t \sim U[0, 1], X_0, X_1} \left[ \| v_\theta(X_t, t) - (X_1 - X_0) \|^2 \right]$$


In [ ]:
import torch
import torch.nn as nn

class FlowMatchingTrainer:
    def __init__(self, model):
        self.model = model
        
    def loss(self, x1): # x1 is real data
        B = x1.shape[0]
        x0 = torch.randn_like(x1) # Gaussian noise
        t = torch.rand(B, 1, 1, 1, device=x1.device) # Uniform time t in [0, 1]
        
        # Straight line trajectory
        xt = (1.0 - t) * x0 + t * x1
        target_velocity = x1 - x0
        
        predicted_velocity = self.model(xt, t.squeeze())
        return F.mse_loss(predicted_velocity, target_velocity)

print("Flow Matching Objective: Simple, elegant, and straight trajectories!")
